### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="infrared_thermography_temperature",
    dataset_year="2023",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://physionet.org/content/face-oral-temp-data/1.0.0/",
    download_description="""
    In this notebook:

    from ucimlrepo import fetch_ucirepo 
    import pandas as pd
    import os 
    infrared_thermography_temperature = fetch_ucirepo(id=925)  
    X = infrared_thermography_temperature.data.features 
    y = infrared_thermography_temperature.data.targets 

    if not os.path.exists("../../../local-data-warehouse/infrared_thermography_temperature"):
        os.makedirs("../../../local-data-warehouse/infrared_thermography_temperature")

    pd.concat([X,y], axis=1).to_csv("../../../local-data-warehouse/infrared_thermography_temperature/data.csv", index=False)
""",
    # References
    academic_reference_bibtex="""@article{wang2023facial,
  title={Facial and oral temperature data from a large set of human subject volunteers},
  author={Wang, Quanzeng and Zhou, Yangling and Ghassemi, Pejman and Chenna, Dwith and Chen, Michelle and Casamento, Jon and Pfefer, Joshua and Mcbride, David},
  journal={PhysioNet, May},
  year={2023}
}

""",
    academic_reference_bibtex_key="wang2023facial",
    license="Creative Commons Zero 1.0 Universal Public Domain Dedication",
    data_tags=["IID"],
    curation_comments="""
    - We use the temperature measured in monitor mode (aveOralM) as target because fast mode (aveOralF) is stated to be less accurate.
    - We exclude T_atm, Humidity, and Distance since they are control parameters from the lab scenario and would not be available in a real-world deployment scenario.
    - We exclude T_offset1, because it is a derived feature that is not available in a real-world deployment scenario.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="aveOralM",
    problem_type="regression",
    objective_metric_name="rmse",
    # stratify_on="",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "data.csv")
df = df.drop(columns=["aveOralF", "T_atm", "Humidity", "Distance", "T_offset1"])
print("Loaded data shape:", df.shape)

Loaded data shape: (1020, 30)


In [3]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,Gender,Age,Ethnicity,Max1R13_1,Max1L13_1,aveAllR13_1,aveAllL13_1,T_RC1,T_RC_Dry1,T_RC_Wet1,T_RC_Max1,T_LC1,T_LC_Dry1,T_LC_Wet1,T_LC_Max1,RCC1,LCC1,canthiMax1,canthi4Max1,T_FHCC1,T_FHRC1,T_FHLC1,T_FHBC1,T_FHTC1,T_FH_Max1,T_FHC_Max1,T_Max1,T_OR1,T_OR_Max1,aveOralM
0,Male,41-50,White,35.0300,35.3775,34.4000,34.9175,34.9850,34.9850,34.7625,35.0325,35.3375,35.3375,34.4850,35.3775,34.7850,34.4650,35.3775,35.3375,33.5775,33.4775,33.3725,33.4925,33.0025,34.5300,34.0075,35.6925,35.6350,35.6525,36.59
1,Female,31-40,Black or African-American,34.5500,34.5200,33.9300,34.2250,34.7100,34.6325,34.6400,34.7425,34.5600,34.5375,34.3500,34.5750,34.3225,34.2400,34.7400,34.7150,34.0325,34.0550,33.6775,33.9700,34.0025,34.6825,34.6600,35.1750,35.0925,35.1075,37.19
2,Female,21-30,White,35.6525,35.5175,34.2775,34.8000,35.6850,35.6675,35.6150,35.7175,35.5025,35.5025,35.2950,35.5300,35.3575,35.0925,35.7175,35.6825,34.9000,34.8275,34.6475,34.8200,34.6700,35.3450,35.2225,35.9125,35.8600,35.8850,37.34
3,Female,21-30,Black or African-American,35.2225,35.6125,34.3850,35.2475,35.2075,35.2000,35.1175,35.2250,35.5950,35.5950,35.3275,35.6125,34.9100,35.1700,35.6125,35.5950,34.4400,34.4225,34.6550,34.3025,34.9175,35.6025,35.3150,35.7200,34.9650,34.9825,37.09
4,Male,18-20,White,35.5450,35.6650,34.9100,35.3675,35.6025,35.4750,35.5700,35.6400,35.6400,35.6400,35.0775,35.6675,35.3550,35.1200,35.6650,35.6475,35.0900,35.1600,34.3975,34.6700,33.8275,35.4175,35.3725,35.8950,35.5875,35.6175,37.04


## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,020
Columns: 30
Use sampling: False (sample size: 1,020)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['aveAllR13_1', 'T_FHTC1', 'T_FHLC1', 'T_FHRC1', 'T_FHCC1', 'T_FHBC1', 'aveAllL13_1', 'LCC1', 'T_FHC_Max1', 'RCC1']
Rows remaining as candidates after top-10 filter: 0 (of 1,020)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,Gender,Age,Ethnicity,Max1R13_1,Max1L13_1,aveAllR13_1,aveAllL13_1,T_RC1,T_RC_Dry1,T_RC_Wet1,T_RC_Max1,T_LC1,T_LC_Dry1,T_LC_Wet1,T_LC_Max1,RCC1,LCC1,canthiMax1,canthi4Max1,T_FHCC1,T_FHRC1,T_FHLC1,T_FHBC1,T_FHTC1,T_FH_Max1,T_FHC_Max1,T_Max1,T_OR1,T_OR_Max1,aveOralM
0,Male,41-50,White,35.0300,35.3775,34.4000,34.9175,34.9850,34.9850,34.7625,35.0325,35.3375,35.3375,34.4850,35.3775,34.7850,34.4650,35.3775,35.3375,33.5775,33.4775,33.3725,33.4925,33.0025,34.5300,34.0075,35.6925,35.6350,35.6525,36.59
1,Female,31-40,Black or African-American,34.5500,34.5200,33.9300,34.2250,34.7100,34.6325,34.6400,34.7425,34.5600,34.5375,34.3500,34.5750,34.3225,34.2400,34.7400,34.7150,34.0325,34.0550,33.6775,33.9700,34.0025,34.6825,34.6600,35.1750,35.0925,35.1075,37.19
2,Female,21-30,White,35.6525,35.5175,34.2775,34.8000,35.6850,35.6675,35.6150,35.7175,35.5025,35.5025,35.2950,35.5300,35.3575,35.0925,35.7175,35.6825,34.9000,34.8275,34.6475,34.8200,34.6700,35.3450,35.2225,35.9125,35.8600,35.8850,37.34
3,Female,21-30,Black or African-American,35.2225,35.6125,34.3850,35.2475,35.2075,35.2000,35.1175,35.2250,35.5950,35.5950,35.3275,35.6125,34.9100,35.1700,35.6125,35.5950,34.4400,34.4225,34.6550,34.3025,34.9175,35.6025,35.3150,35.7200,34.9650,34.9825,37.09
4,Male,18-20,White,35.5450,35.6650,34.9100,35.3675,35.6025,35.4750,35.5700,35.6400,35.6400,35.6400,35.0775,35.6675,35.3550,35.1200,35.6650,35.6475,35.0900,35.1600,34.3975,34.6700,33.8275,35.4175,35.3725,35.8950,35.5875,35.6175,37.04


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Max1R13_1,float64,0.0,0.0,581.0,"35.48, 35.6775, 35.5375, 35.415, 35.705, 35.2375, 35.4925, 35.8375, 35.385, 35.2825"
1,Max1L13_1,float64,0.0,0.0,567.0,"35.4, 35.665, 35.6225, 35.6625, 35.6, 35.8425, 35.9225, 35.27, 35.91, 35.5775"
2,aveAllR13_1,float64,0.0,0.0,670.0,"34.9575, 35.305, 35.175, 34.735, 34.77, 34.96, 34.95, 34.4325, 34.5625, 35.54"
3,aveAllL13_1,float64,0.0,0.0,628.0,"34.9975, 34.8775, 34.87, 34.55, 34.98, 35.35, 34.925, 34.86, 34.8, 34.7425"
4,T_RC1,float64,0.0,0.0,592.0,"35.37, 35.435, 35.4775, 35.3725, 35.795, 35.4225, 35.7275, 35.6575, 35.6025, 35.665"
5,T_RC_Dry1,float64,0.0,0.0,589.0,"35.5775, 35.23, 35.4475, 35.3425, 35.5875, 35.9575, 35.365, 35.32, 35.475, 35.6725"
6,T_RC_Wet1,float64,0.0,0.0,575.0,"35.25, 35.3775, 35.6975, 35.29, 35.955, 35.605, 35.285, 35.6475, 35.3625, 35.3125"
7,T_RC_Max1,float64,0.0,0.0,577.0,"35.54, 35.515, 35.6825, 35.7675, 35.3875, 35.53, 35.4375, 35.775, 35.9575, 35.7075"
8,T_LC1,float64,0.0,0.0,570.0,"35.86, 35.595, 35.32, 35.29, 35.635, 35.5825, 35.245, 35.38, 35.7325, 35.2"
9,T_LC_Dry1,float64,0.0,0.0,578.0,"35.34, 35.2925, 35.9275, 35.47, 35.32, 35.4925, 35.6125, 35.3675, 36.0625, 35.575"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Max1R13_1,1020.0,35.596533,0.574888,33.8975,38.4050
Max1L13_1,1020.0,35.611474,0.549760,34.1225,38.0425
aveAllR13_1,1020.0,34.888475,0.718613,31.7700,37.5750
aveAllL13_1,1020.0,35.011345,0.633836,32.9025,37.6800
T_RC1,1020.0,35.659921,0.553897,33.9850,38.3850
T_RC_Dry1,1020.0,35.587143,0.569278,33.8250,38.3800
T_RC_Wet1,1020.0,35.547315,0.568828,33.9325,38.3300
T_RC_Max1,1020.0,35.689762,0.553594,34.0025,38.4075
T_LC1,1020.0,35.640851,0.541169,34.1050,38.0425
T_LC_Dry1,1020.0,35.610869,0.545645,34.1050,38.0375


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column    rank                                         
Age       1                         18-20    534  52.35
          2                         21-25    355  34.80
          3                         26-30     67   6.57
          4                         31-40     31   3.04
          5                         51-60     11   1.08
Ethnicity 1                         White    506  49.61
          2                         Asian    260  25.49
          3     Black or African-American    143  14.02
          4               Hispanic/Latino     57   5.59
          5                   Multiracial     50   4.90
Gender    1                        Female    606  59.41
          2                          Male    414  40.59

In [9]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,1.976,1.878,0.26,0.0,log,51565.0,170003.8,exponential


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=10, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019d006d-5845-7aef-ac6a-597e3831a6ba
70cd1edbcdb5e13ce101c2ed52ceabd923f8a350f6196183afecc749fd121329
